# AllRecipes Data Analysis Project

## 1. Introduction
*(We will introduce the data set and our guiding question here.)*

## 2. Data Cleaning and Summary
*(We will document our transformations and summary statistics here.)*

### 2.1 Imports

In [14]:
import pandas as pd
import numpy as np
from scipy import stats

In [15]:
# Load data
url_recipes = 'https://raw.githubusercontent.com/rfordatascience/tidytuesday/main/data/2025/2025-09-16/all_recipes.csv'
url_cuisines = 'https://raw.githubusercontent.com/rfordatascience/tidytuesday/main/data/2025/2025-09-16/cuisines.csv'

# Turn links into DataFrames
df_recipes = pd.read_csv(url_recipes)
df_cuisines = pd.read_csv(url_cuisines)

df_cuisines.head()

,name,country,url,author,date_published,ingredients,calories,fat,carbs,protein,avg_rating,total_ratings,reviews,prep_time,cook_time,total_time,servings
0,Saganaki (Flaming Greek Cheese),Greek,https://www.allrecipes.com/recipe/263750/flami...,John Mitzewich,2024-02-07,"1 (4 ounce) package kasseri cheese, 1 tablespo...",391.0,25.0,15.0,16.0,4.8,25.0,22.0,10,5,15,2.0
1,Coney Island Knishes,Jewish,https://www.allrecipes.com/recipe/272334/coney...,John Mitzewich,2024-11-26,"2 ¾ cups all-purpose flour, or more as needed,...",301.0,17.0,31.0,7.0,4.6,10.0,9.0,30,75,180,16.0
2,Diana's Hawaiian Bread Rolls,Australian and New Zealander,https://www.allrecipes.com/recipe/22797/dianas...,CHIPPENDALE,2022-07-14,1 ½ cups warm water (110 degrees F/45 degrees ...,64.0,3.0,9.0,1.0,4.3,126.0,104.0,20,15,180,12.0
3,Chilean Pebre,Chilean,https://www.allrecipes.com/recipe/273763/chile...,Heidi,2025-01-31,"½ cup chopped cilantro, ¼ cup olive oil, ¼ cup...",106.0,9.0,7.0,1.0,5.0,1.0,1.0,10,0,10,6.0
4,Tex-Mex Sheet Cake,Tex-Mex,https://www.allrecipes.com/recipe/22388/tex-me...,Ann,2025-02-18,"2 cups all-purpose flour, 1 ½ cups brown sugar...",449.0,23.0,58.0,7.0,3.8,13.0,11.0,30,15,45,15.0


In [16]:
df_recipes.head()

,name,url,author,date_published,ingredients,calories,fat,carbs,protein,avg_rating,total_ratings,reviews,prep_time,cook_time,total_time,servings
0,Chewy Whole Wheat Peanut Butter Brownies,https://www.allrecipes.com/recipe/140717/chewy...,DMOMMY,2020-06-18,"⅓ cup margarine, softened, ⅔ cup white sugar, ...",222.0,13.0,24.0,6.0,4.4,47.0,36.0,20,35,55,16.0
1,Pumpkin Pie Eggnog,https://www.allrecipes.com/recipe/269204/pumpk...,Bobbie Susan,2022-09-26,"12 egg yolks, 2 cups heavy whipping cream, ½ ...",477.0,31.0,43.0,8.0,5.0,1.0,1.0,10,5,495,8.0
2,Eggs Poached in Tomato Sauce,https://www.allrecipes.com/recipe/238054/eggs-...,Bren,2018-06-08,"2 tablespoons olive oil, or to taste, ½ onion...",354.0,18.0,32.0,20.0,4.8,4.0,4.0,10,75,85,4.0
3,Minestrone Casserole,https://www.allrecipes.com/minestrone-casserol...,Sarah Brekke,2025-03-03,4 cups dried mafalda pasta (mini lasagna noodl...,356.0,9.0,53.0,19.0,4.3,14.0,13.0,20,40,60,8.0
4,Yummy Stuffed Peppers,https://www.allrecipes.com/recipe/241937/yummy...,Procrastigirl,2024-12-11,"4 green bell peppers, halved lengthwise and se...",366.0,22.0,23.0,19.0,4.7,84.0,67.0,30,95,125,8.0


### Data cleaning

In [17]:
# Use outer join to merge cuisine labels onto recipes
df = pd.merge(df_recipes, 
              df_cuisines[['url', 'country']].rename(columns={'country': 'cuisine'}), 
              on='url', 
              how='outer')

# Standardize column types to numeric
num_cols = ["calories", "fat", "carbs", "protein", "avg_rating", 
            "total_ratings", "reviews", "prep_time", "cook_time", 
            "total_time", "servings"]

for col in num_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Drop rows missing essential metrics
df.dropna(subset=['avg_rating', 'ingredients', 'prep_time', 
                  'cook_time', 'total_time'], inplace=True)

# Remove recipes with 0 time
df = df[df['total_time'] > 0]

### Feature Engineering

In [18]:
# Count ingredient lines
def count_ingredients(s):
    if pd.isna(s): return 0
    return len([item for item in str(s).split(",") if item.strip()])

df['n_ingredient_lines'] = df['ingredients'].apply(count_ingredients)

In [19]:
# Actionable Score
df_filtered = df[df['total_ratings'] >= 10].copy() # Filter for reliable averages
df_filtered['z_rating'] = stats.zscore(df_filtered['avg_rating'])

df_filtered['z_ingredients'] = stats.zscore(df_filtered['n_ingredient_lines'])
df_filtered['complexity_score'] = df_filtered['z_rating'] - df_filtered['z_ingredients']

df_filtered['z_log_time'] = stats.zscore(np.log1p(df_filtered['total_time']))
df_filtered['action_score'] = df_filtered['z_rating'] - df_filtered['z_log_time']

df_filtered['ultimate_score'] = df_filtered['z_rating'] - (df_filtered['z_log_time'] + df_filtered['z_ingredients'])

### Key summary statistics

In [20]:
print("--- General Dataset Shape ---")
print(f"Total recipes after cleaning: {len(df):,}")
print(f"Recipes with cuisine tags: {df['cuisine'].notna().sum():,}")

print("\n--- Central Tendencies for Time and Complexity ---")
time_stats = df[['prep_time', 'cook_time', 'total_time', 'n_ingredient_lines']].describe().loc[['mean', '50%', 'std']]
display(time_stats)

print("\n--- User Interaction Summary ---")
rating_stats = df[['avg_rating', 'total_ratings']].describe().loc[['mean', '50%', 'max']]
display(rating_stats)

print("\n--- Correlation: Time vs. Rating ---")
corr_time_rating = df[['total_time', 'avg_rating']].corr().iloc[0,1]
print(f"Pearson Correlation between Total Time and Average Rating: {corr_time_rating:.4f}")

print("\n--- Top 5 Most Actionable Recipes (High Rating, Low Time) ---")
display(df_filtered.sort_values('action_score', ascending=False)[['name', 'total_time', 'avg_rating', 'n_ingredient_lines', 'action_score']].head(5))

print("--- Top 5: Simplest High-Rated Recipes (Low Ingredients) ---")
display(df_filtered.sort_values('complexity_score', ascending=False)
        [['name', 'avg_rating', 'n_ingredient_lines', 'complexity_score']].head(5))

print("\n--- Top 5: Ultimate Actionable (High Rating, Low Time, Low Ingredients) ---")
display(df_filtered.sort_values('ultimate_score', ascending=False)
        [['name', 'avg_rating', 'total_time', 'n_ingredient_lines', 'ultimate_score']].head(5))

print("\n--- Cuisine Distribution (Top 10) ---")
display(df['cuisine'].value_counts().head(10))

--- General Dataset Shape ---
Total recipes after cleaning: 13,104
Recipes with cuisine tags: 1,001

--- Central Tendencies for Time and Complexity ---


,prep_time,cook_time,total_time,n_ingredient_lines
mean,17.790140,44.717415,149.599435,12.390644
50%,15.000000,25.000000,55.000000,11.000000
std,23.601901,99.632413,901.598716,5.725828



--- User Interaction Summary ---


,avg_rating,total_ratings
mean,4.531219,102.082036
50%,4.600000,26.000000
max,5.000000,997.000000



--- Correlation: Time vs. Rating ---
Pearson Correlation between Total Time and Average Rating: 0.0080

--- Top 5 Most Actionable Recipes (High Rating, Low Time) ---


,name,total_time,avg_rating,n_ingredient_lines,action_score
8077,Christmas Poinsettia Punch,5.0,5.0,9,4.022274
1761,Peach Cosmo,5.0,5.0,5,4.022274
7213,Prickly Pear Margarita,5.0,5.0,6,4.022274
2227,Simple Cranberry Spinach Salad,5.0,5.0,5,4.022274
4417,Pumpkin Pie Spice I,1.0,4.7,4,3.979555


... (2 outputs left out)


--- Top 5: Ultimate Actionable (High Rating, Low Time, Low Ingredients) ---


,name,avg_rating,total_time,n_ingredient_lines,ultimate_score
14909,Classic White Wine Spritzer,4.9,5.0,2,5.483654
4417,Pumpkin Pie Spice I,4.7,1.0,4,5.442965
1761,Peach Cosmo,5.0,5.0,5,5.301106
2227,Simple Cranberry Spinach Salad,5.0,5.0,5,5.301106
5676,Classic Ham Glaze,4.9,5.0,3,5.299076



--- Cuisine Distribution (Top 10) ---


cuisine
Chinese             64
Indian              64
Jewish              59
Italian             57
Korean              55
Cajun and Creole    53
Tex-Mex             52
Spanish             39
Soul Food           35
French              32
Name: count, dtype: int64

In [21]:
# 1. Nutritional Profile of Actionable vs. Non-Actionable
# Let's define "Top Actionable" as the top 10% of ultimate_score
top_threshold = df_filtered['ultimate_score'].quantile(0.9)
df_filtered['is_top_actionable'] = df_filtered['ultimate_score'] >= top_threshold

print("--- Nutritional Comparison: Actionable vs. Others ---")
# Do faster, highly-rated recipes have more sugar/fat (calories)?
nutritional_comp = df_filtered.groupby('is_top_actionable')[['calories', 'fat', 'protein']].median()
display(nutritional_comp)

# 2. Author Concentration: Are "Actionable" recipes dominated by specific authors?
print("\n--- Authors of the Most Actionable Recipes ---")
top_authors = df_filtered[df_filtered['is_top_actionable']]['author'].value_counts().head(5)
display(top_authors)

# 3. Complexity vs. Time Distribution
# Does more time usually mean more ingredients? 
complexity_corr = df[['total_time', 'n_ingredient_lines']].corr().iloc[0,1]
print(f"\nCorrelation between Total Time and Number of Ingredients: {complexity_corr:.4f}")

# 4. Review Volatility
# Are "Actionable" recipes more controversial? (Standard deviation of reviews)
print("\n--- Review Engagement by Time Category ---")
df['time_quartile'] = pd.qcut(df['total_time'], 4, labels=['Very Fast', 'Fast', 'Medium', 'Slow'])
engagement = df.groupby('time_quartile', observed=True)['total_ratings'].agg(['mean', 'std'])
display(engagement)

--- Nutritional Comparison: Actionable vs. Others ---


,calories,fat,protein
is_top_actionable,,,
False,317.0,15.0,9.0
True,181.0,9.0,4.0



--- Authors of the Most Actionable Recipes ---


author
John Mitzewich       30
Allrecipes Member    16
France Cevallos      12
Sara                  9
Carol                 7
Name: count, dtype: int64


Correlation between Total Time and Number of Ingredients: 0.0048

--- Review Engagement by Time Category ---


,mean,std
time_quartile,,
Very Fast,100.252870,172.070778
Fast,103.902074,176.484700
Medium,110.125816,181.118917
Slow,93.493643,162.234260


## 3. Visualisation
*(Code for our three plots will go here.)*

## 4. Discussions
*(We will explain the insights from our plots here.)*

## 5. Teamwork
*(Tasks allocation paragraph goes here.)*


## 6. References
* TidyTuesday AllRecipes Dataset: https://github.com/rfordatascience/tidytuesday/blob/main/data/2025/2025-09-16/readme.md